# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [2]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU (UUID: GPU-a2f32238-420d-3493-bafe-90f1e473bebc)


In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [5]:
!python scripts/prepare_data.py --kaggle

Dataset URL: https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
License(s): MIT

dataset_version : d10-21e4070301
  train  1422 ảnh  boxes={'pothole': 1074, 'crack': 1750, 'manhole': 691}
  val     276 ảnh  boxes={'pothole': 66, 'crack': 470, 'manhole': 103}
  test    311 ảnh  boxes={'pothole': 121, 'crack': 299, 'manhole': 163}



  0%|          | 0.00/185M [00:00<?, ?B/s]
  1%|          | 1.00M/185M [00:00<01:41, 1.90MB/s]
  1%|          | 2.00M/185M [00:00<00:58, 3.27MB/s]
  2%|▏         | 4.00M/185M [00:00<00:33, 5.61MB/s]
  3%|▎         | 5.00M/185M [00:01<00:29, 6.32MB/s]
  4%|▍         | 7.00M/185M [00:01<00:22, 8.23MB/s]
  5%|▍         | 9.00M/185M [00:01<00:20, 9.02MB/s]
  5%|▌         | 10.0M/185M [00:01<00:20, 8.83MB/s]
  6%|▌         | 11.0M/185M [00:01<00:21, 8.54MB/s]
  7%|▋         | 13.0M/185M [00:01<00:17, 10.5MB/s]
  8%|▊         | 15.0M/185M [00:02<00:17, 10.0MB/s]
  9%|▊         | 16.0M/185M [00:02<00:20, 8.86MB/s]
  9%|▉         | 17.0M/185M [00:02<00:19, 8.89MB/s]
 10%|█         | 19.0M/185M [00:02<00:20, 8.46MB/s]
 11%|█         | 20.0M/185M [00:02<00:21, 8.20MB/s]
 11%|█▏        | 21.0M/185M [00:02<00:20, 8.23MB/s]
 12%|█▏        | 22.0M/185M [00:02<00:20, 8.18MB/s]
 12%|█▏        | 23.0M/185M [00:03<00:22, 7.48MB/s]
 13%|█▎        | 24.0M/185M [00:03<00:23, 7.25MB/s]
 14%|█▍        | 26.

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# Chay duoc khi thu muc hien tai la thu muc lab hoac thu muc cha.
lab_dir = Path.cwd()
if not (lab_dir / "data/splits/train.jsonl").exists():
    lab_dir = lab_dir / "K4-Track-4-Day10-VisionLanguageModel"
data_dir = lab_dir / "data"

splits = {}
for split in ("train", "val", "test"):
    with (data_dir / "splits" / f"{split}.jsonl").open(encoding="utf-8") as f:
        splits[split] = [json.loads(line) for line in f if line.strip()]
    counts = Counter(obj["label"] for row in splits[split] for obj in row["objects"])
    print(f"{split}: {len(splits[split])} images, {sum(counts.values())} boxes | {dict(counts)}")

# Moi phien quay chi duoc xuat hien trong mot tap du lieu.
sessions = {name: {row["session"] for row in rows} for name, rows in splits.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = sessions[a] & sessions[b]
    assert not overlap, f"Overlapping sessions in {a}/{b}: {sorted(overlap)}"
print("OK: train/val/test have no overlapping sessions.")

train_records = splits["train"]
print("\nFirst 3 train records:")
for row in train_records[:3]:
    print(json.dumps(row, ensure_ascii=False, indent=2))

# Chon 6 anh co nhan; seed giup lan chay sau chon lai cung cac anh.
labeled_records = [row for row in train_records if row["objects"]]
if not labeled_records:
    raise ValueError("No labeled images found in train.jsonl")
samples = random.Random(42).sample(labeled_records, min(6, len(labeled_records)))
colors = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax in axes.flat:
    ax.axis("off")

for ax, row in zip(axes.flat, samples):
    with Image.open(data_dir / row["image_path"]) as image:
        image = image.convert("RGB")
    width, height = image.size
    ax.imshow(image)
    for obj in row["objects"]:
        # YOLO: tam (xc, yc), rong w, cao h deu nam tren thang 0-1.
        # Doi sang toa do pixel de ve khung len anh.
        xc, yc, w, h = obj["yolo"]
        x = (xc - w / 2) * width
        y = (yc - h / 2) * height
        box_width, box_height = w * width, h * height
        color = colors.get(obj["label"], "lime")
        ax.add_patch(Rectangle((x, y), box_width, box_height,
                               fill=False, edgecolor=color, linewidth=2))
        ax.text(x, y, obj["label"], color=color, fontsize=10,
                va="bottom", bbox=dict(facecolor="black", alpha=0.6, edgecolor="none"))
    ax.set_title(f"{row['image_id']} | {len(row['objects'])} boxes", fontsize=9)

plt.tight_layout()
plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9